In [61]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
import scipy.constants as pc
import astropy.units as u
from astropy.io import fits
from astropy.coordinates import SkyCoord
from astropy.table import Table
from astropy.table import vstack
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import ttest_rel
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import AutoMinorLocator
import os
from matplotlib.gridspec import GridSpec
import h5py
import emcee
import corner

FIG_DIR = "/raid/scratch/work/alberttg/Masters_project/Plots"

DATA_PRODUCTS_PATH = "/raid/scratch/work/alberttg/Masters_project/Data_products"

POSTERIOR_PATH = '/raid/scratch/work/austind/GALFIND_WORK/Bagpipes/pipes/posterior'

In [62]:
def get_posterior(sample_table, SFH_model):
    """
    Finding the paths to the stellar mass posterior, based on galaxy survey id, survey and SFH model used in BAGPIPES.
    Each galaxy has its own posterior.
    """

    surveys = np.asarray(sample_table["SURVEY"].data, dtype=str)

    survey_ids = np.asarray(sample_table["#ID"].data, dtype=int)

    redshifts = np.asarray(sample_table["input_redshift"].data, dtype=float)

    mass = np.asarray(sample_table["stellar_mass_50"].data, dtype=float)

    src_corr = np.asarray(sample_table["ext_src_corr_0.32as_F444W"].data, dtype=float)

    posteriors = []

    for i in range(len(sample_table)):

        gal_id = survey_ids[i]
        survey = surveys[i]
        redshift = redshifts[i]
        corr = src_corr[i]

        if survey in ["CEERSP1", "CEERSP2", "CEERSP3", "CEERSP4", "CEERSP5", "CEERSP6", "CEERSP7", "CEERSP8", "CEERSP9", "CEERSP10", "NEP-1", "NEP-2", "NEP-3", "NEP-4", "NGDEEP"]:

            path = os.path.join(POSTERIOR_PATH, f"v14_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{gal_id}.h5")

        elif survey in ["PRIMER-COSMOS","PRIMER-UDS"]:
                
            path = os.path.join(POSTERIOR_PATH, f"v12_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{gal_id}.h5")


        elif survey in ["JADES-DR3-GS-East", "JADES-DR3-GS-North", "JADES-DR3-GS-South", "JADES-DR3-GS-West", "JADES-DR3-GN-Parallel", "JADES-DR3-GN-Medium", "JADES-DR3-GN-Deep"]:

            path = os.path.join(POSTERIOR_PATH, f"v13_psfmatch_F444W_empirical/{survey}/ACS_WFC+NIRCam/0.32as/{SFH_model}/{gal_id}.h5")
            
        else:
            print(f"Galaxy {gal_id} could not be found in these surveys")
            print(f"Check survey {survey}")
            continue


        with h5py.File(path,'r') as hf:
            # Print all root level object names (aka keys) 
            # these can be group or dataset names 
            # print("Keys: %s" % hf['basic_quantities'].keys())
            # print("Keys: %s" % hf['advanced_quantities'].keys())
            stellar_mass = hf['basic_quantities']['stellar_mass'][:]
            
        stellar_mass = stellar_mass * corr
        median_mass = np.median(stellar_mass)
        log_mass = np.array(stellar_mass - median_mass)

        if len(log_mass) != 500:
            print("Length not 500: ", len(log_mass))
            print(f"Check the posterior of galaxy {gal_id} in survey {survey} ")

        posteriors.append(log_mass) # ought to be in same order as mass and redshift are ordered, which is by ith row of sample table

    # print(posteriors)
    
    plot_binned_posterior(mass, redshifts, posteriors, 60) # I think the posterior are in the same order as galaxy table (mass, redshift...)


In [63]:
def calculate_fwhm(centres, counts):
    """
    Calculate the Full Width at Half Maximum (FWHM)
    directly from a histogram using linear interpolation.

    Parameters
    ----------
    centres : array
        Histogram bin centres.
    counts : array
        Histogram heights (e.g. normalised density).

    Returns
    -------
    fwhm : float
        Full width at half maximum.
    left_crossing : float
        Left half-maximum crossing position.
    right_crossing : float
        Right half-maximum crossing position.
    """

    centres = np.asarray(centres, dtype=float)
    counts = np.asarray(counts, dtype=float)

    # Find the peak of the histogram
    peak_idx = np.argmax(counts) # index of the data associated with the heighest peak
    peak_height = counts[peak_idx]

    # if peak_height <= 0:
    #    return np.nan, np.nan, np.nan # for safety

    # Calculate half maximum
    half_max = peak_height / 2.0

    # Find left crossing
    left_idx = np.where(counts[:peak_idx] < half_max)[0] # the indices of everything to the left of the peak

    if len(left_idx) == 0:
        left_crossing = np.nan
    else:
        i = left_idx[-1] # the index of the point just closest to the left side fwhh
        left_crossing = np.interp(
            half_max,
            [counts[i], counts[i+1]],
            [centres[i], centres[i+1]]
        ) # linearly interpolate because the data point probably isnt actually at half the peak

    # Find right crossing, same idea
    right_idx = np.where(counts[peak_idx+1:] < half_max)[0]


    if len(right_idx) == 0:
        right_crossing = np.nan
    else:
        i = peak_idx + 1 + right_idx[0] - 1
        right_crossing = np.interp(
            half_max,
            [counts[i+1], counts[i]],
            [centres[i+1], centres[i]]
        )

    # Calculate FWHM
    fwhm = right_crossing - left_crossing
    print("right crossing: ", right_crossing)
    print("left crossing: ", left_crossing)
    print("fwhm: ", fwhm)
    
    return fwhm, left_crossing, right_crossing

In [64]:
def plot_binned_posterior(mass, redshift, posterior, histogram_bins):
    """
    Sorting the posterior into bins, selected by mass and redshift value.

    mass : array of floats
        The mass values of all the galaxies
    redshift : array of floats
        The zfix redshift values of all the galaxies
    posterior : array of arrays of floats
        The posterior for each galaxy, which are each an array.
    histogram_bins : float
        Number of histogram bins for each plot and also for gaussian fitting
    """
    z_bins = [[6.5, 7.5], [7.5, 8.5], [8.5, 9.5], [9.5, 10.5], [10.5, 11.5], [11.5, 13.5]]

    fig, axes = plt.subplots(3, 6, figsize=(30,20), constrained_layout=True)
    # Starting binning, bin by mass first (vertical bins)
    # mass_bins = np.linspace(mass.min(), mass.max(), n_mass_bins + 1)
    # mass_bins = [[mass_bins[i], mass_bins[i + 1]] for i in range(len(mass_bins) - 1)]
    # mass_bins.reverse()
    #print("Mass bins: ", mass_bins)
    mass_bins = [[9,11], [7,9], [6,7]] # Have put them in reverse for the gridspec plotting

    for i in range(len(mass_bins)):
        
        bin_width = mass_bins[i][1] - mass_bins[i][0]

        mass_bin_mask = (mass >= mass_bins[i][0]) & (mass < mass_bins[i][1]) # Splitting into mass bins

        redshift_m_binned = redshift[mass_bin_mask]
        mass_errors_m_binned = mass[mass_bin_mask]
        
        posterior_m_binned = [item for item, m in zip(posterior, mass_bin_mask) if m]


        # Now bin by z (horizontal bins)
        for j in range(len(z_bins)):
            
            z_bin_mask = (redshift_m_binned >= z_bins[j][0]) & (redshift_m_binned < z_bins[j][1])

            redshift_z_binned = redshift_m_binned[z_bin_mask]
            mass_z_binned = mass_errors_m_binned[z_bin_mask] # x axis values now binned by z and mass

            print("m bin: ", mass_bins[i])
            print("z bin: ", z_bins[j])
            print("Number of gals in bin: ", len(redshift_z_binned))

            posterior_z_binned = [item for item, m in zip(posterior_m_binned, z_bin_mask) if m]

            # [print(len(p)) for p in posterior_z_binned] # this checks that most posterior are length 500, but one is 106 and one is 107

            if len(posterior_z_binned) == 0:
                continue  # or leave the axis empty

            # --------------------------- binning complete ----------------------------
            # Now calculating posterior sum and making into a normalised histogram

            posterior_sum = np.concatenate(posterior_z_binned, axis=0)  # was trying this to see combined distribution of mass offsets for the galaxies in a bin
            # posterior_sum = [sum(p) for p in zip(*posterior_z_binned)] # idk about this one...
            # print("sum shape: ", np.shape(posterior_sum))
            
            raw_counts, edges = np.histogram(posterior_sum, bins=histogram_bins)
            centres = 0.5 * (edges[1:] + edges[:-1])
            yerr = np.sqrt(np.maximum(raw_counts, 1)) # Poisson errors on hist density

            # Calculate the normalization factor used by density=True
            bin_widths = np.diff(edges)
            total_counts = len(posterior_sum)
            normalization = total_counts * bin_widths

            # Calculate density and its corresponding scaled error
            counts = raw_counts / normalization
            # Apply Poisson error on raw counts, safely handling zero-bins with np.maximum
            yerr = np.sqrt(np.maximum(raw_counts, 1)) / normalization

            # Calculate fwhm
            fwhm, left_crossing, right_crossing = calculate_fwhm(centres, counts) # idk how to use the poisson error on counts
            peak_height = np.max(counts)
            half_max = peak_height / 2      

            # ----------------------------------- Plotting ------------------------------------
            
            ax = axes[i,j]
                    
            # ax.set_ylabel(r'Probability density \left( $\sigma_{M_\star}^{-1}$ \right)', fontsize=16)
            if j == 0:
                ax.set_ylabel(rf"${mass_bins[i][0]} ≤ \log_{10}(M_\star/M_\odot) < {mass_bins[i][1]}$", fontsize=16)
            
            ax.grid(True)
    
            # ax.errorbar(x, y, xerr=[xe_lo, xe_hi], yerr=[ye_lo, ye_hi], fmt=".k", alpha=0.4, label="Data", zorder=1)
            
            ax.tick_params(
                    axis="both",
                    which="major",
                    labelsize=14,
                    length=8,
                    width=1.5,
                    direction="in",
                    top=True,
                    right=True
                    )
            ax.xaxis.set_minor_locator(AutoMinorLocator(5))
            ax.yaxis.set_minor_locator(AutoMinorLocator(5))
            ax.tick_params(
                    axis="both",
                    which="minor",
                    length=4,
                    width=1,
                    direction="in",
                    top=True,
                    right=True
                    )

            ax.hist(posterior_sum, bins=histogram_bins, color='steelblue', density=True)

            ax.axhline(half_max, color='black', linestyle='--',
            label='Half maximum')

            ax.axvline(x=left_crossing, ymin=0, ymax=0.5, color='red', linestyle=':')
            ax.axvline(right_crossing,  ymin=0, ymax=0.5, color='red', linestyle=':')

            ax.fill_between(
                [left_crossing, right_crossing],
                [0, 0],
                [half_max, half_max],
                alpha=0.3,
                color='red',
                label=f'FWHM = {fwhm:.3f}'
            )

            ax.legend(loc="upper left")

            legend_text = rf"${z_bins[j][0]} ≤ z < {z_bins[j][1]}$"
            ax.text(0.6, 0.95, legend_text, transform=ax.transAxes, fontsize=15, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.5))

    fig.supylabel(r'Histogram density', fontsize=20)
    fig.supxlabel(r'Stellar Mass $\log_{10}(M_\star/M_\odot) - \langle( \log_{10}(M_\star/M_\odot) \rangle $', fontsize=20)
    plt.tight_layout()
    fig_path = os.path.join(FIG_DIR, "gridspec_mass_errors.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


In [65]:
if __name__ == "__main__":

    with fits.open('/raid/scratch/work/alberttg/Masters_project/Data_products/First_sample_table.fits') as hdu:

        sample_table = hdu[1].data
    sample_table = Table(sample_table)

    SFH_models = ["Bagpipes_sfh_cont_bursty_zEAZYsfhzblueagn_3.0,10.0Myr_Calzetti_log_10_Z_log_10_BPASS_zfix"]

    get_posterior(sample_table, SFH_models[0])

Length not 500:  106
Check the posterior of galaxy 11825 in survey NEP-4 
Length not 500:  117
Check the posterior of galaxy 96942 in survey PRIMER-UDS 
m bin:  [9, 11]
z bin:  [6.5, 7.5]
Number of gals in bin:  50
right crossing:  0.06421710868666815
left crossing:  -0.0657094080463266
fwhm:  0.12992651673299477
m bin:  [9, 11]
z bin:  [7.5, 8.5]
Number of gals in bin:  10
right crossing:  0.069364990126289
left crossing:  -0.06553988660858984
fwhm:  0.13490487673487883
m bin:  [9, 11]
z bin:  [8.5, 9.5]
Number of gals in bin:  6
right crossing:  0.047794455841617844
left crossing:  -0.04554180355176204
fwhm:  0.09333625939337989
m bin:  [9, 11]
z bin:  [9.5, 10.5]
Number of gals in bin:  5
right crossing:  0.04476782634916771
left crossing:  -0.035184765598822165
fwhm:  0.07995259194798987
m bin:  [9, 11]
z bin:  [10.5, 11.5]
Number of gals in bin:  14
right crossing:  0.10453522161460854
left crossing:  -0.07014293614259179
fwhm:  0.17467815775720033
m bin:  [9, 11]
z bin:  [11.5, 1

/tmp/ipykernel_1442292/3353752544.py:139: UserWarning: The figure layout has changed to tight
  plt.tight_layout()
